# P10. GAN-LPN PRNG: generator pseudolosowy wspierany przez GenAI

Notatnik przygotowany na podstawie pracy: Xuguang Wu, Yiliang Han, Minqing Zhang, Shuaishuai Zhu, Xu An Wang, *Generative artificial intelligence-driven secure pseudorandom number generator*, Future Generation Computer Systems 182 (2026), 108492, DOI: 10.1016/j.future.2026.108492.

Celem jest przedstawienie zasady działania generatora bez wchodzenia w pełne dowody kryptograficzne. W poprzednich notatnikach zwykle po opisie pojawiała się implementacja i walidacja. Tutaj status implementacji jest inny, bo generator zależy od wytrenowanego modelu sieci neuronowej.


## Dopasowanie do poprzednich notatników

Struktura tego notatnika jest podobna do P7-P9: najpierw krótka motywacja, potem opis mechanizmu, parametry i ograniczenia praktyczne. Różnica polega na tym, że w poprzednich przykładach generatory dało się zaimplementować bez dodatkowych artefaktów treningowych. W tej pracy rdzeń PRNG jest prosty do zapisania, ale źródło szumu jest wytrenowanym modelem GAN, więc samo przepisanie wzoru nie odtwarza generatora z publikacji.


## Status implementacji

**Nie przygotowuję lokalnej implementacji do testów statystycznych w tym notatniku.**

Powód jest metodologiczny, nie tylko techniczny. Generator opisany w pracy składa się z dwóch części: wytrenowanego samplera BNES opartego na WGAN-GP oraz rdzenia LPN. Publiczne repozytorium autorów jest dostępne pod adresem https://gitee.com/guangandy/ganlpnprng i zawiera kod PaddlePaddle, ale sprawdzona wersja repozytorium nie zawiera wytrenowanego pliku wag `gan_model_full_output_run_0_1024.pdparams`, którego adapter używa do generowania bitów. Lokalne środowisko projektu nie zawiera też PaddlePaddle.

Zastąpienie BNES zwykłym `Bernoulli(p)`, hashem albo małą siecią losowo zainicjalizowaną dałoby jedynie demonstrację idei, a nie generator z pracy. Dlatego poniżej jest opis i kontrola warunków uruchomienia, bez implementacji opartej na domysłach.


## Główna idea

Autorzy próbują połączyć dwie rzeczy:

- **GAN / WGAN-GP**: model uczy się produkować wektor szumu podobny do próbek z rozkładu Bernoulliego o zadanym prawdopodobieństwie jedynki, w pracy około `p = 0.2`.
- **LPN, czyli Learning Parity with Noise**: problem kryptograficzny, w którym obserwujemy liniowe równania nad bitami, ale część wyników jest zakłócona szumem. Odzyskanie sekretu jest trudne, gdy parametry i szum są dobrane poprawnie.

Sama sieć neuronowa nie jest tutaj traktowana jako pełna podstawa bezpieczeństwa. Jej rola jest bardziej ograniczona: ma dostarczać szum o kontrolowanych własnościach. Bezpieczeństwo ma wynikać z rdzenia LPN, a niedoskonałości samplera są w pracy ujmowane jako dodatkowe składniki pogarszające oszacowanie bezpieczeństwa.


## Składnik 1: BNES, czyli sampler szumu Bernoulliego

BNES (*Bernoulli Noise Expansion Sampler*) dostaje krótki seed i zwraca dłuższy wektor bitów. W instancji z pracy seed ma długość `l = 128`, a wektor szumu ma długość `n = 1024`.

Najprostsza intuicja: model ma działać jak deterministyczna maszyna, która z krótkiego wejścia tworzy 1024 bity wyglądające jak niezależne próbki Bernoulliego z prawdopodobieństwem jedynki bliskim `0.2`. Żeby to uzyskać, autorzy trenują generator i dyskryminator w wariancie WGAN-GP. Generator jest binarizowany na wyjściu przez BinarySTE: w przód daje bity `0/1`, a podczas uczenia pozwala przepuścić przybliżony gradient.

Dyskryminator nie sprawdza tylko, czy próbka wygląda losowo. Ma też głowy oceniające średnią i zależności między bitami, bo zbyt duże korelacje osłabiłyby założenie LPN.


## Składnik 2: rdzeń LPN generatora

Generator utrzymuje stan złożony z dwóch części:

- `v_t`: tajny stan liniowy, w pracy `m = 320` bitów,
- `r_t`: seed dla BNES, w pracy `l = 128` bitów.

W każdej rundzie BNES tworzy wektor szumu `e_t`. Następnie generator liczy liniową transformację stanu i miesza ją z tym szumem operacją XOR:

```text
c_t = M^T v_t XOR e_t
```

Wektor `c_t` jest potem dzielony na trzy części: nowy `v`, nowy `r` oraz bity wyjściowe `z_t`. Dla parametrów z tabeli w pracy (`n = 1024`, `m = 320`, `l = 128`) jedna runda daje `mu = n - m - l = 576` bitów wyjściowych.

Można to czytać tak: poprzedni stan wyznacza część strukturalną, BNES dodaje kontrolowany szum, a wynik jednocześnie aktualizuje stan i produkuje porcję bitów.


## Algorytm w skrócie

Poniżej jest pseudokod zredukowany do najważniejszych operacji. To nie jest implementacja wykonywalna, tylko opis przepływu danych.

```text
Dane: macierz M, seed = v_0 || r_0

dla kolejnych rund t:
    e_t = BNES(r_t)                 # wytrenowany GAN tworzy szum Bernoulliego
    c_t = M^T v_t XOR e_t           # rdzeń LPN

    v_{t+1} = pierwsze m bitów c_t
    r_{t+1} = kolejne l bitów c_t
    z_t     = pozostałe bity c_t

    zwróć z_t
```

Ważna uwaga: kod autorów w publicznym repozytorium zawiera wariant, w którym `total_output_dim = n * m`, czyli znacznie większy wektor roboczy niż w tabeli parametrów opisanej w artykule. To dodatkowy powód, żeby nie tworzyć lokalnej implementacji przez zgadywanie, który wariant należy uznać za kanoniczny do testów w tym repozytorium.


In [ ]:
from pathlib import Path
import importlib.util

asset_candidates = [Path("P10-assets"), Path("noteboooks/P10-assets")]
asset_dir = next((p for p in asset_candidates if p.exists()), asset_candidates[0])
paper = asset_dir / "Generative_artificial_intelligence-driven_secure_pseudorandom_number_generator.pdf"

model_files = sorted(asset_dir.glob("*.pdparams")) + sorted(asset_dir.glob("*.pdmodel"))
has_paddle = importlib.util.find_spec("paddle") is not None

print("PDF w P10-assets:", paper.exists())
print("PaddlePaddle dostępny w tym kernelu:", has_paddle)
print("Lokalne pliki wag/modelu:", [p.name for p in model_files] or "brak")

if not has_paddle or not model_files:
    print("Wniosek: nie uruchamiam generatora GAN-LPN w tym notatniku.")
    print("Brakuje zależności PaddlePaddle i/lub wytrenowanych wag BNES.")


## Co byłoby potrzebne do przebadania generatora

Aby dodać implementację zgodną z tym projektem, potrzebne byłyby co najmniej:

- działające środowisko PaddlePaddle zgodne z kodem autorów,
- wytrenowane wagi BNES albo uruchomienie pełnego treningu z parametrami z pracy,
- decyzja, czy testujemy wariant z tabeli artykułu (`n = 1024`) czy wariant z repozytorium autorów (`total_output_dim = n * m`),
- adapter zgodny z interfejsem używanym w poprzednich notatnikach: `random_uint64`, `random_bits`, `random_bytes`, `random_floats`.

Dopiero wtedy można byłoby użyć tych samych testów co wcześniej, np. `validate_generator(...)` i wizualizacji z `run_visual_tests(...)`.


## Wnioski

- Konstrukcja pasuje tematycznie do wcześniejszych notatników o generatorach kryptograficznych, bo wykorzystuje klasyczny twardy problem LPN.
- Nowością jest użycie generatywnej AI nie jako samodzielnego PRNG, ale jako samplera szumu dla rdzenia LPN.
- Sama idea działania jest jasna: BNES tworzy szum, rdzeń LPN miesza go ze stanem, a wynik aktualizuje stan i daje bity wyjściowe.
- Nie ma tu rzetelnej lokalnej implementacji do testów, ponieważ bez wytrenowanego BNES i zgodnego środowiska PaddlePaddle testowalny kod byłby innym generatorem niż ten opisany w pracy.

Referencje: artykuł PDF w `P10-assets/` oraz publiczne repozytorium autorów `https://gitee.com/guangandy/ganlpnprng`.
